# Benchmark 1: 2 Class vs 4 Class: Cross-Session

In [9]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
import scipy
from pyriemann.tangentspace import TangentSpace

In [10]:
from pyriemann.utils import mean_riemann
from scipy.optimize import minimize
from pymanopt import Problem
from pymanopt.manifolds import SpecialOrthogonalGroup
from pymanopt.optimizers import SteepestDescent
from pymanopt import Problem
from functools import partial
from pymanopt.function import numpy as pymanopt_numpy
import autograd.numpy as anp  # Autograd's NumPy replacement
from autograd import grad
import pymanopt
import autograd.scipy.linalg as linalg
from pyriemann.classification import MDM
import numpy as np
from pyriemann.estimation import Covariances
from pyriemann.utils.mean import mean_riemann
from scipy.linalg import fractional_matrix_power, logm, eigh
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
from sklearn.neighbors import NearestNeighbors

In [11]:
active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772}
active_lr_event_ids = {'769': 769, '770': 770}
unknown_event_id = {'783': 783} 

In [12]:
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2a'

In [13]:
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [14]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [15]:
# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.
sfreq = 250
tmin = 0.5       # Epoch start at cue onset.
# Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
tmax = 3.5  # This gives 4.0 seconds.
filter_order = 50

In [16]:
def twofour_crosssession(n_classes):

    if(n_classes==2):
        event_ids = active_lr_event_ids
    else:
        event_ids = active_all_event_ids
    

    train_active_X = []         # List to hold numpy arrays with shape (n_trials, 22, 1001) per subject.
    train_active_y = []         # List to hold event labels per subject.
    train_active_metadata = []  # List to hold event metadata per subject.

    # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
    for subj in range(1, 10):
        filename = os.path.join(data_dir, f'A{subj:02d}T.gdf')
        
        # Read the GDF file (using preload=True to load data into memory).
        train_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        # Retain only EEG channels (22 channels) and exclude EOG channels.
        train_raw.pick_types(eeg=True, eog=False)
        
        # Extract events corresponding only to the four desired types.
        train_active_events, _ = mne.events_from_annotations(train_raw, event_id=event_ids)
        
        # Create epochs from tmin to tmax.
        train_active_epochs = mne.Epochs(train_raw, train_active_events, event_id=event_ids, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)
        
        # Get the epoch data (num_epochs x 22 channels x 1001 samples).
        train_active_data = train_active_epochs.get_data()
        
        # Print the number of extracted epochs to verify
        print(f"Subject {subj}: Epoch data shape {train_active_data.shape}")
        
        # Sampling frequency from raw.info (should be 250).
        fs = int(train_raw.info['sfreq'])
        # print(fs)
        n_trials, n_channels, n_times = train_active_data.shape
        train_active_filtered_data = np.empty_like(train_active_data)
        
        # Apply the causal bandpass filter channel‐wise for each trial.
        for trial in range(n_trials):
            for ch in range(n_channels):
                train_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    train_active_data[trial, ch, :],
                    lowcut=8,    # Lower bound of sensorimotor rhythm.
                    highcut=30,  # Upper bound of sensorimotor rhythm.
                    fs=fs,
                    order=filter_order     # Lower order for a smoother causal filter.
                )
        # Apply the causal bandpass filter channel‐wise for each trial.
        # for trial in range(n_trials):
        #     for ch in range(n_channels):
        #         train_active_filtered_data[trial, ch, :] = train_active_data[trial, ch, :]
        
        # Append the processed data, labels, and event metadata.
        train_active_X.append(train_active_filtered_data)
        train_active_y.append(train_active_epochs.events[:, 2])  # The third column holds the event code.
        train_active_metadata.append(train_active_epochs.events)

    print("Loaded data for", len(train_active_X), "subjects.")

    eval_active_X = []         
    eval_active_y = []        
    eval_active_metadata = [] 

    # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
    for subj in range(1, 10):
        filename = os.path.join(data_dir, f'A{subj:02d}E.gdf')
        mat_data = loadmat(f'/home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels/A{subj:02d}E.mat')
        true_y =  np.array(mat_data['classlabel'], dtype=np.int64).reshape(288,) + 768
        # Read the GDF file (using preload=True to load data into memory).
        eval_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        # Retain only EEG channels (22 channels) and exclude EOG channels.
        eval_raw.pick_types(eeg=True, eog=False)
        
        # Extract events corresponding only to the four desired types.
        eval_active_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id)
        
        # Create epochs from tmin to tmax.
        eval_active_epochs = mne.Epochs(eval_raw, eval_active_events, event_id=unknown_event_id, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)
        
        # Get the epoch data (num_epochs x 22 channels x 1001 samples).
        eval_active_data = eval_active_epochs.get_data()
        
        # Print the number of extracted epochs to verify
        print(f"Subject {subj}: Epoch data shape {eval_active_data.shape}")
        
        # Sampling frequency from raw.info (should be 250).
        fs = int(eval_raw.info['sfreq'])
        # print(fs)
        n_trials, n_channels, n_times = eval_active_data.shape
        eval_active_filtered_data = np.empty_like(eval_active_data)
        
        # Apply the causal bandpass filter channel‐wise for each trial.
        for trial in range(n_trials):
            for ch in range(n_channels):
                eval_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    eval_active_data[trial, ch, :],
                    lowcut=8,    # Lower bound of sensorimotor rhythm.
                    highcut=30,  # Upper bound of sensorimotor rhythm.
                    fs=fs,
                    order=filter_order     # Lower order for a smoother causal filter.
                )
        
        # for trial in range(n_trials):
        #     for ch in range(n_channels):
        #         eval_active_filtered_data[trial, ch, :] = eval_active_data[trial, ch, :]
                    
        
        # Append the processed data, labels, and event metadata.
        eval_active_X.append(eval_active_filtered_data)
        eval_active_y.append(true_y)  # The third column holds the event code.
        eval_active_metadata.append(eval_active_epochs.events)

    print("Loaded data for", len(eval_active_X), "subjects.")

    if(n_classes==2):
        eval_active_X = [x[np.isin(y, [769, 770])] for x, y in zip(eval_active_X, eval_active_y)]
        eval_active_y = [y[np.isin(y, [769, 770])] for y in eval_active_y]
    
    train_active_y = encode_labels(train_active_y)
    eval_active_y = encode_labels(eval_active_y)

        # Initialize list to hold tangent space features for each subject
    train_aligned = []

    # Loop over all 9 subjects
    for subj in range(9):
        # Compute covariance matrices for each trial (shape: 144, 22, 22)
        cov_est = Covariances(estimator='scm')
        P = cov_est.fit_transform(train_active_X[subj])  # Input: (144, 22, 501), Output: (144, 22, 22)

        ts = TangentSpace(metric='riemann')
        X_ts = ts.fit_transform(P)  # Returned shape: (n_trials, feature_dim); for 22 channels, feature_dim is 253
        
        train_aligned.append(X_ts.T)

    eval_aligned = []

    # Loop over all 9 subjects
    for subj in range(9):
        # Compute covariance matrices for each trial (shape: 144, 22, 22)
        cov_est = Covariances(estimator='scm')
        P = cov_est.fit_transform(train_active_X[subj])  # Input: (144, 22, 501), Output: (144, 22, 22)

        ts = TangentSpace(metric='riemann')
        X_ts = ts.fit_transform(P)  # Returned shape: (n_trials, feature_dim); for 22 channels, feature_dim is 253
        
        eval_aligned.append(X_ts.T)

    align_per_class = 14
    n_subjects = 9
    if(n_classes==2):
        classes = [0, 1]
    else:
        classes = [0, 1, 2, 3]  # Two classes as per your setup
    epsilon = 1e-6
    n_channels = 22
    N = 5
    alpha = 0.01
    beta = 0.1
    rho = 20
    p = 10
    d = 253
    accuracies = []

    for subj_idx in range(len(train_active_X)):
        # print(subj_idx)
        # Split data into train/test using leave-one-subject-out
        X_target = eval_aligned[subj_idx]
        y_target = eval_active_y[subj_idx]
        
        # Concatenate data from other subjects
        X_source = train_aligned[subj_idx]
        y_source = train_active_y[subj_idx]

        n_s = X_source.shape[1]  # 1152
        n_t = X_target.shape[1]  # 144

        # Initial pseudo-labels
        clf_init = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
        clf_init.fit(X_source.T, y_source)
        hat_y_t = np.zeros_like(clf_init.predict(X_target.T))

        # Iterative optimization
        for n in range(N):
            # Source domain scatter matrices
            m_0 = X_source[:, y_source == 0].mean(axis=1)  # Shape: (253,)
            m_1 = X_source[:, y_source == 1].mean(axis=1)
            if (n_classes==4):  # Shape: (253,)
                m_2 = X_source[:, y_source == 2].mean(axis=1)  # Shape: (253,)
                m_3 = X_source[:, y_source == 3].mean(axis=1)
            m = X_source.mean(axis=1)  # Shape: (253,)
            n_0 = np.sum(y_source == 0)
            n_1 = np.sum(y_source == 1)
            if(n_classes==4):
                n_2 = np.sum(y_source == 2)
                n_3 = np.sum(y_source == 3)
            S_b = n_0 * np.outer(m_0 - m, m_0 - m) + n_1 * np.outer(m_1 - m, m_1 - m)  # Shape: (253, 253)
            if (n_classes==4):
                S_b = n_0 * np.outer(m_0 - m, m_0 - m) + n_1 * np.outer(m_1 - m, m_1 - m) + n_2 * np.outer(m_2 - m, m_2 - m) + n_3 * np.outer(m_3 - m, m_3 - m)
            S_w = np.zeros((d, d))  # Shape: (253, 253)
            for k_class in classes:
                X_k = X_source[:, y_source == k_class]  # Shape: (253, n_k)
                S_w_k = np.cov(X_k, rowvar=True) * (X_k.shape[1] - 1)  # Shape: (253, 253)
                S_w += S_w_k

            # Target domain similarity matrix
            nn = NearestNeighbors(n_neighbors=10)
            nn.fit(X_target.T)
            distances, indices_nn = nn.kneighbors(X_target.T)
            sigma = 1.0
            S = np.zeros((n_t, n_t))  # Shape: (144, 144)
            for i in range(n_t):
                for j in indices_nn[i]:
                    if j != i:
                        S[i, j] = np.exp(-np.sum((X_target[:, i] - X_target[:, j])**2) / (2 * sigma**2))
                        S[j, i] = S[i, j]

            # Laplacian matrix
            D = np.diag(np.sum(S, axis=1))
            D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D)))
            # L_target = np.eye(n_t) - D_inv_sqrt @ S @ D_inv_sqrt
            L_target = D - S
            # Shape: (144, 144)

            # Centering matrix
            H = np.eye(n_t) - np.ones((n_t, n_t)) / n_t  # Shape: (144, 144)

            # One-hot encodings
            Y_s = np.eye(n_classes)[y_source]  # Shape: (1152, 2)
            hat_Y_t = np.eye(n_classes)[hat_y_t]  # Shape: (144, 2)
            N_s = Y_s / n_s  # Shape: (1152, 2)
            # N_t = hat_Y_t / n_t  # Shape: (144, 2)
            N_t = np.zeros((n_t, n_classes)) if n == 0 else np.eye(n_classes)[hat_y_t] / n_t

            # Joint probability MMD matrix R
            R11 = X_source @ N_s @ N_s.T @ X_source.T  # Shape: (253, 253)
            R12 = -X_source @ N_s @ N_t.T @ X_target.T  # Shape: (253, 253)
            R21 = -X_target @ N_t @ N_s.T @ X_source.T  # Shape: (253, 253)
            R22 = X_target @ N_t @ N_t.T @ X_target.T  # Shape: (253, 253)
            R = np.block([[R11, R12], [R21, R22]])  # Shape: (506, 506)

            # Other matrices
            P = np.block([[S_w, np.zeros((d, d))], [np.zeros((d, d)), np.zeros((d, d))]])  # Shape: (506, 506)
            L = np.block([[np.zeros((d, d)), np.zeros((d, d))], [np.zeros((d, d)), X_target @ L_target @ X_target.T]])  # Shape: (506, 506)
            I_d = np.eye(d)
            U = np.block([[I_d, -I_d], [-I_d, 2 * I_d]])  # Shape: (506, 506)
            V = np.block([[np.zeros((d,d)), np.zeros((d,d))], [np.zeros((d,d)), X_target @ H @ X_target.T]]) # Shape: (506, 506) - 

            # Optimization
            A = alpha * P + beta * L + rho * U + R  # Shape: (506, 506)
            B = V + 1e-6 * np.eye(2 * d)  # Shape: (506, 506)
            eigvals, eigvecs = eigh(A, B)
            W = eigvecs[:, :p]  # Shape: (506, 10)
            A_proj = W[:d, :]  # Shape: (253, 10)
            B_proj = W[d:, :]  # Shape: (253, 10)

            # Train and predict
        
            clf = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
            clf.fit((A_proj.T @ X_source).T, y_source)
            hat_y_t = clf.predict((B_proj.T @ X_target).T)

        # Compute accuracy
        acc = balanced_accuracy_score(y_target, hat_y_t)
        accuracies.append(acc)
    
    for subj_idx in range(len(train_active_X)):
        print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

    print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
    

        
    

In [18]:
twofour_crosssession(2)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 1: Epoch data shape (144, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 2: Epoch data shape (144, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 7: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 8: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Epoch data shape (288, 22, 751)
Loaded data for 9 subjects.
Subject 1 Test Accuracy: 0.51
Subject 2 Test Accuracy: 0.65
Subject 3 Test Accuracy: 0.97
Subject 4 Test Accuracy: 0.55
Subject 5 Test Accuracy: 0.65
Subject 6 Test Accuracy: 0.44
Subject 7 Test Accuracy: 0.81
Subject 8 Test Accuracy: 0.97
Subject 9 Test Accuracy: 0.54

Mean Cross-Validation Accuracy: 0.68 ± 0.19


In [19]:
twofour_crosssession(4)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 1: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 2: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 7: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 8: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Epoch data shape (288, 22, 751)
Loaded data for 9 subjects.
Subject 1 Test Accuracy: 0.21
Subject 2 Test Accuracy: 0.67
Subject 3 Test Accuracy: 0.89
Subject 4 Test Accuracy: 0.23
Subject 5 Test Accuracy: 0.43
Subject 6 Test Accuracy: 0.23
Subject 7 Test Accuracy: 0.84
Subject 8 Test Accuracy: 0.91
Subject 9 Test Accuracy: 0.22

Mean Cross-Validation Accuracy: 0.51 ± 0.29
